# 04 — Initial reports and checks

**Learner scaffold · RetailFlow dataset 1.0.0**

Course lessons: Day 2, Lesson 8.

Goal: Build initial reports, validate every expected view, and persist Gold snapshots.

Before running: Run 03 through persistence; Silver must still contain 60 sales.

Writes: <prefix>_gold_initial_* tables (eight checked views). Other stages’ tables remain separate.

Follow the playlist module order. Build Buddy will help you create the missing numbered exercise cells. The instructor has a separate completed reference. Step IDs are stable anchors for that authoring work. Saved outputs are blank unless explicitly verified locally; a passed local check does not imply Databricks execution.


## Your notebook grows through Build Buddy

Configuration and instructor utility cells are supplied so you can focus on the learning task. The numbered exercise cells have no solutions. Read the playlist instructions, attempt each cell, run it, explain its output, then ask for a hint if needed. A blank exercise cell is incomplete work. Do not use Run All expecting this scaffold to be a finished pipeline. The course revisits notebooks: Lesson 5 runs 02-02, 02-03 and 02-05; Lesson 6 then runs 02-04.


## Configuration

Use the same assigned names in every notebook. Each notebook defines its own variables and helpers; it does not depend on another notebook’s Python memory. Persistent tables are the handoff between notebooks.


In [ ]:
# Edit these four values in EVERY notebook before running it.
CATALOG = "your_catalog"
SCHEMA = "your_training_schema"
VOLUME = "your_training_volume"
TABLE_PREFIX = "rf_yourname"

import re
for name in [CATALOG, SCHEMA, VOLUME, TABLE_PREFIX]:
    assert re.fullmatch(r"[a-z][a-z0-9_]*", name), "Use lowercase letters, digits, and underscores; start with a letter."
assert not any(name.startswith("your_") for name in [CATALOG, SCHEMA, VOLUME]), "Replace the example names with your assigned names."
assert TABLE_PREFIX != "rf_yourname", "Choose the table prefix assigned to you."
DATASET_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/retailflow-v1.0.0"
print("Dataset:", DATASET_ROOT)
print("Table prefix:", f"{CATALOG}.{SCHEMA}.{TABLE_PREFIX}_")


## Supplied support cells

These utilities keep repeated setup and checks in one place inside this notebook. The Python cleaner deliberately processes this tiny fixture on the driver so beginners can inspect each row. Spark handles DataFrames, joins, summaries, and Delta tables. This is a teaching pattern for 80 sales, not a large-scale ingestion design. The helper definitions below are part of the notebook and can be inspected. Learners will construct selected parts through Build Buddy in the later authoring phase.


In [ ]:
from pathlib import Path
from decimal import Decimal, ROUND_HALF_UP, InvalidOperation
from datetime import date, datetime
from collections import Counter, defaultdict
import csv, json, re

def read_rows(relative_path):
    with (Path(DATASET_ROOT) / relative_path).open(encoding='utf-8', newline='') as stream:
        return list(csv.DictReader(stream))

def money(value):
    return Decimal(value).quantize(Decimal('0.01'), rounding=ROUND_HALF_UP)

def business_payload(row):
    return {key: value for key, value in row.items() if key != 'source_row_id'}

def business_records(rows):
    return sorted([business_payload(row) for row in rows], key=lambda row: row['sale_id'])

def dimension_maps():
    return (
        {r['store_id']: r for r in read_rows('data/stores.csv')},
        {r['product_id']: r for r in read_rows('data/products.csv')},
        {r['customer_id']: r for r in read_rows('data/customers.csv')},
    )

def clean_sales(raw_rows):
    """Readable Python cleaning for this tiny fixture; no distributed UDF required."""
    stores, products, customers = dimension_maps()
    candidates, rejected, repaired = [], [], []
    for raw in sorted(raw_rows, key=lambda row: row['source_row_id']):
        row = {key: value.strip() for key, value in raw.items()}
        for key in ['store_id', 'product_id', 'customer_id', 'payment_method']:
            row[key] = row[key].upper()
        if row['customer_id'] == 'UNKNOWN':
            row['customer_id'] = ''
        reason = None
        if any(not row[k] for k in ['sale_id', 'source_row_id', 'sale_date', 'store_id', 'product_id']):
            reason = 'REQUIRED_MISSING'
        if not reason:
            try:
                fmt = '%d/%m/%Y' if '/' in row['sale_date'] else '%Y-%m-%d'
                pattern = r'\d{2}/\d{2}/\d{4}' if '/' in row['sale_date'] else r'\d{4}-\d{2}-\d{2}'
                if not re.fullmatch(pattern, row['sale_date']):
                    raise ValueError('Date format mismatch')
                row['sale_date'] = datetime.strptime(row['sale_date'], fmt).date().isoformat()
            except ValueError:
                reason = 'DATE_INVALID'
        if not reason:
            if row['product_id'] not in products:
                reason = 'PRODUCT_UNKNOWN'
            elif row['store_id'] not in stores:
                reason = 'STORE_UNKNOWN'
            elif row['customer_id'] and row['customer_id'] not in customers:
                reason = 'CUSTOMER_UNKNOWN'
        if not reason:
            try:
                if not re.fullmatch(r'\d+', row['quantity']) or int(row['quantity']) < 1:
                    reason = 'QUANTITY_INVALID'
                elif not re.fullmatch(r'\d+', row['record_version']) or int(row['record_version']) < 1:
                    reason = 'VERSION_INVALID'
                else:
                    row['unit_price'] = format(money(row['unit_price'] or products[row['product_id']]['list_price']), '.2f')
                    row['discount_pct'] = row['discount_pct'] or '0'
                    price, discount = Decimal(row['unit_price']), Decimal(row['discount_pct'])
                    if not price.is_finite() or price <= 0:
                        reason = 'PRICE_INVALID'
                    elif not discount.is_finite() or not 0 <= discount <= 100:
                        reason = 'DISCOUNT_INVALID'
                    elif row['payment_method'] not in ['UPI', 'CARD', 'CASH']:
                        reason = 'PAYMENT_INVALID'
            except (ValueError, InvalidOperation):
                reason = 'NUMBER_INVALID'
        if reason:
            rejected.append(dict(raw, rejection_reason=reason))
        else:
            if row != raw:
                repaired.append(row['source_row_id'])
            candidates.append(row)
    groups = defaultdict(list)
    for row in candidates:
        groups[(row['sale_id'], row['record_version'])].append(row)
    accepted, duplicate_ids = [], []
    raw_by_id = {row['source_row_id']: row for row in raw_rows}
    for group in groups.values():
        if any(business_payload(row) != business_payload(group[0]) for row in group):
            rejected.extend(dict(raw_by_id[row['source_row_id']], rejection_reason='VERSION_CONFLICT') for row in group)
        else:
            accepted.append(group[0])
            duplicate_ids.extend(row['source_row_id'] for row in group[1:])
    return accepted, rejected, duplicate_ids, repaired

def current_versions(rows):
    state = {}
    for row in rows:
        old = state.get(row['sale_id'])
        if old is None or int(row['record_version']) > int(old['record_version']):
            state[row['sale_id']] = row
    return sorted(state.values(), key=lambda row: row['sale_id'])

def visible_returns(stage, sales_rows):
    rows = [r for r in read_rows('data/returns.csv') if stage == 'final' or r['available_batch'] == '01']
    sales = {row['sale_id']: row for row in sales_rows}
    assert len({r['return_id'] for r in rows}) == len(rows)
    returned = Counter()
    for row in rows:
        sale = sales[row['sale_id']]
        assert row['available_batch'] in ['01', '02']
        assert date.fromisoformat(row['return_date']) >= date.fromisoformat(sale['sale_date'])
        quantity = int(row['return_quantity'])
        assert quantity > 0
        returned[row['sale_id']] += quantity
        assert returned[row['sale_id']] <= int(sale['quantity'])
        assert Decimal(row['refund_amount']) == money(quantity * Decimal(sale['unit_price']) * (1 - Decimal(sale['discount_pct']) / 100))
    return rows


In [ ]:
from pyspark.sql import functions as F, types as T

SALES_COLUMNS = ['source_row_id', 'sale_id', 'sale_date', 'store_id', 'product_id', 'customer_id', 'quantity', 'unit_price', 'discount_pct', 'payment_method', 'record_version']
SALES_SCHEMA = T.StructType([
    T.StructField(name, T.DateType() if name == 'sale_date' else
                  T.IntegerType() if name in ['quantity', 'record_version'] else
                  T.DecimalType(12, 2) if name == 'unit_price' else
                  T.DecimalType(5, 2) if name == 'discount_pct' else T.StringType(), True)
    for name in SALES_COLUMNS
])

def sales_frame(rows):
    typed = []
    for row in rows:
        values = []
        for name in SALES_COLUMNS:
            value = row[name]
            if name == 'sale_date': value = date.fromisoformat(value)
            elif name in ['quantity', 'record_version']: value = int(value)
            elif name in ['unit_price', 'discount_pct']: value = Decimal(value)
            elif name == 'customer_id' and value == '': value = None
            values.append(value)
        typed.append(tuple(values))
    return spark.createDataFrame(typed, SALES_SCHEMA)

def raw_frame(relative_path):
    return spark.read.option('header', True).option('inferSchema', False).option('mode', 'FAILFAST').csv(f'{DATASET_ROOT}/{relative_path}')

def table_name(suffix):
    assert re.fullmatch(r'[a-z][a-z0-9_]*', suffix)
    return f'{CATALOG}.{SCHEMA}.{TABLE_PREFIX}_{suffix}'

def assert_expected(frame, relative_path, ignore_provenance=False):
    columns = [c for c in frame.columns if not (ignore_provenance and c == 'source_row_id')]
    actual = frame.select(*columns)
    # Read full files before dropping provenance so schema fields match CSV positions.
    expected = spark.read.schema(frame.schema).option('header', True).option('mode', 'FAILFAST').csv(f'{DATASET_ROOT}/checks/expected/{relative_path}').select(*columns)
    assert actual.exceptAll(expected).limit(1).count() == 0, f'Extra/different rows: {relative_path}'
    assert expected.exceptAll(actual).limit(1).count() == 0, f'Missing rows: {relative_path}'
    print('PASS:', relative_path)

def collected_sales(frame):
    rows = []
    for record in frame.select(*SALES_COLUMNS).collect():
        row = record.asDict()
        row['sale_date'] = row['sale_date'].isoformat()
        row['quantity'] = str(row['quantity'])
        row['record_version'] = str(row['record_version'])
        row['unit_price'] = format(row['unit_price'], '.2f')
        row['discount_pct'] = format(row['discount_pct'].normalize(), 'f')
        row['customer_id'] = row['customer_id'] or ''
        rows.append(row)
    return sorted(rows, key=lambda row: row['sale_id'])

def merge_counts(current, incoming):
    joined = incoming.alias('s').join(current.alias('t'), F.col('s.sale_id') == F.col('t.sale_id'), 'left')
    business_columns = [c for c in SALES_COLUMNS if c != 'source_row_id']
    equal = F.lit(True)
    for name in business_columns:
        equal = equal & F.col(f's.{name}').eqNullSafe(F.col(f't.{name}'))
    conflicts = joined.where((F.col('s.record_version') == F.col('t.record_version')) & ~equal)
    assert conflicts.limit(1).count() == 0, 'Equal-version business conflict: investigate before merge.'
    counts = {'insert': 0, 'update': 0, 'noop': 0}
    for row in joined.select(
        F.when(F.col('t.sale_id').isNull(), 'insert')
         .when(F.col('s.record_version') > F.col('t.record_version'), 'update')
         .otherwise('noop').alias('action')
    ).groupBy('action').count().collect():
        counts[row['action']] = row['count']
    return counts

def merge_into_silver(incoming):
    from delta.tables import DeltaTable
    assert incoming.groupBy('sale_id').count().where('count > 1').limit(1).count() == 0
    target = table_name('silver_sales')
    counts = merge_counts(spark.table(target), incoming)
    (DeltaTable.forName(spark, target).alias('t')
        .merge(incoming.alias('s'), 't.sale_id = s.sale_id')
        .whenMatchedUpdateAll(condition='s.record_version > t.record_version')
        .whenNotMatchedInsertAll().execute())
    return counts


In [ ]:
AMOUNT_COLUMNS = ['gross_sales', 'discount_amount', 'net_sales', 'refund_amount', 'net_revenue']

def return_frame(rows):
    schema = 'return_id string, sale_id string, return_date date, return_quantity int, refund_amount decimal(12,2), reason string, available_batch string'
    values = [(r['return_id'], r['sale_id'], date.fromisoformat(r['return_date']), int(r['return_quantity']), Decimal(r['refund_amount']), r['reason'], r['available_batch']) for r in rows]
    return spark.createDataFrame(values, schema)

def sales_with_amounts(sales, returns):
    refunds = returns.groupBy('sale_id').agg(F.sum('return_quantity').alias('returned_units'), F.sum('refund_amount').alias('refund_amount'))
    return (sales
        .withColumn('gross_sales', (F.col('quantity') * F.col('unit_price')).cast('decimal(12,2)'))
        .withColumn('discount_amount', F.round(F.col('gross_sales') * F.col('discount_pct') / F.lit(100), 2).cast('decimal(12,2)'))
        .withColumn('net_sales', (F.col('gross_sales') - F.col('discount_amount')).cast('decimal(12,2)'))
        .join(refunds, 'sale_id', 'left')
        .fillna({'returned_units': 0, 'refund_amount': 0})
        .withColumn('refund_amount', F.col('refund_amount').cast('decimal(12,2)'))
        .withColumn('net_revenue', (F.col('net_sales') - F.col('refund_amount')).cast('decimal(12,2)'))
        .select(*SALES_COLUMNS, 'gross_sales', 'discount_amount', 'net_sales', 'returned_units', 'refund_amount', 'net_revenue'))

def aggregate_lines(lines, keys):
    return lines.groupBy(*keys).agg(
        F.count('sale_id').alias('sale_count'), F.sum('quantity').alias('units_sold'),
        F.sum('gross_sales').alias('gross_sales'), F.sum('discount_amount').alias('discount_amount'),
        F.sum('net_sales').alias('net_sales'), F.sum('returned_units').alias('returned_units'),
        F.sum('refund_amount').alias('refund_amount'), F.sum('net_revenue').alias('net_revenue'))

def report_views(sales, returns):
    lines = sales_with_amounts(sales, returns)
    stores = raw_frame('data/stores.csv').select('store_id', 'store_name')
    products = raw_frame('data/products.csv').select('product_id', 'product_name', 'category')
    totals = aggregate_lines(lines, [])
    row = totals.first().asDict()
    row['net_units'] = row['units_sold'] - row['returned_units']
    row['average_sale_value'] = money(row['net_sales'] / row['sale_count'])
    row['identified_customers'] = sales.select('customer_id').where('customer_id IS NOT NULL').distinct().count()
    row['guest_sales'] = sales.where('customer_id IS NULL').count()
    row['return_events'] = returns.count()
    metric_order = ['sale_count', 'units_sold', 'gross_sales', 'discount_amount', 'net_sales', 'returned_units', 'refund_amount', 'net_revenue', 'net_units', 'average_sale_value', 'identified_customers', 'guest_sales', 'return_events']
    metrics = spark.createDataFrame([(k, format(row[k], '.2f') if isinstance(row[k], Decimal) else str(row[k])) for k in metric_order], 'metric string, value string')
    aggregate_order = ['sale_count', 'units_sold', 'gross_sales', 'discount_amount', 'net_sales', 'returned_units', 'refund_amount', 'net_revenue']
    by_store = stores.join(aggregate_lines(lines, ['store_id']), 'store_id', 'left').fillna(0).select('store_id', 'store_name', *aggregate_order)
    by_product = products.select('product_id', 'product_name').join(aggregate_lines(lines, ['product_id']), 'product_id', 'left').fillna(0).select('product_id', 'product_name', *aggregate_order)
    by_category = aggregate_lines(lines.join(products.select('product_id', 'category'), 'product_id', 'left'), ['category'])
    by_date = aggregate_lines(lines, ['sale_date'])
    sales_daily = lines.groupBy(F.col('sale_date').alias('event_date')).agg(F.sum('net_sales').alias('net_sales'))
    refunds_daily = returns.groupBy(F.col('return_date').alias('event_date')).agg(F.sum('refund_amount').alias('refunds_paid'))
    cash = (sales_daily.join(refunds_daily, 'event_date', 'full').fillna(0)
            .withColumn('cash_movement', F.col('net_sales') - F.col('refunds_paid'))
            .select('event_date', 'net_sales', 'refunds_paid', 'cash_movement'))
    return dict(sales=sales.select(*SALES_COLUMNS), line_amounts=lines, metrics=metrics, by_store=by_store, by_product=by_product, by_category=by_category, by_sale_date=by_date, by_cash_date=cash)


## 04-01: Read the current checkpoint

**Core concept Vibe:** A checkpoint identifies exactly which records belong in a report.

**Build Buddy action:** Read Silver and enable only the returns visible at this stage.

**Expected checkpoint:** 60 current sales and 3 valid return events.


In [ ]:
# Build Buddy step 04-01
# Create your solution here after attempting the task.
# Keep this cell ID; it links your work to the playlist.


## 04-02: Calculate sales and refunds separately

**Core concept Vibe:** Net sales is before refunds; net revenue is after refunds.

**Build Buddy action:** Aggregate returns by sale before joining; inspect the S0001 calculation.

**Expected checkpoint:** S0001 has INR 80.00 net sales and INR 0.00 net revenue.


In [ ]:
# Build Buddy step 04-02
# Create your solution here after attempting the task.
# Keep this cell ID; it links your work to the playlist.


## 04-03: Build and check every report

**Core concept Vibe:** Multiple summaries should reconcile to the same underlying records.

**Build Buddy action:** Create line amounts, metrics, store/product/category/date summaries and compare all eight views.

**Expected checkpoint:** Every initial reference CSV matches; P012 remains visible with zero sales.


In [ ]:
# Build Buddy step 04-03
# Create your solution here after attempting the task.
# Keep this cell ID; it links your work to the playlist.


## 04-04: Distinguish sale dates from refund dates

**Core concept Vibe:** Refund attribution and cash timing answer different questions.

**Build Buddy action:** Compare sale-date net revenue with the cash-date movement view.

**Expected checkpoint:** Both total INR 14184.00; daily values differ because refunds are paid later.


In [ ]:
# Build Buddy step 04-04
# Create your solution here after attempting the task.
# Keep this cell ID; it links your work to the playlist.


## 04-05: Publish checked course Gold tables

**Core concept Vibe:** A saved summary is a reproducible snapshot of a checkpoint.

**Build Buddy action:** Persist each checked view under a stage-specific table name.

**Expected checkpoint:** Eight persisted initial Gold views pass their answer-key comparisons.


In [ ]:
# Build Buddy step 04-05
# Create your solution here after attempting the task.
# Keep this cell ID; it links your work to the playlist.
